In [11]:

from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(".."))


In [12]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


In [13]:
from src.tools import read_calendar, get_customer_profile

tools = {
    "read_calendar": read_calendar,
    "get_customer_profile": get_customer_profile
}


In [14]:
def triage_node(state):
    email = state["email"]
    prompt = f"""
Classify this email into one of:
ignore
notify_human
respond
Email:
{email}
Return only the label.
"""
    label = llm.invoke(prompt).content.strip().lower()
    return {**state, "triage": label}


In [15]:
def react_agent(state):
    email = state["email"]
    prompt = f"""
You are an email assistant.
You can use tools if needed.
Tools:
read_calendar
get_customer_profile
Email:
{email}
If you need a tool, write TOOL:<toolname>
Otherwise give reply.
"""
    response = llm.invoke(prompt).content

    if "TOOL:" in response:
        tool_name = response.replace("TOOL:", "").strip()
        tool_result = tools[tool_name]()
        return {**state, "response": tool_result}

    return {**state, "response": response}


In [16]:
graph = StateGraph(dict)

graph.add_node("triage", triage_node)
graph.add_node("react", react_agent)

def route(state):
    if state["triage"] == "respond":
        return "react"
    else:
        return "end"

graph.add_conditional_edges("triage", route)
graph.set_entry_point("triage")

app = graph.compile()


In [ ]:
emails = pd.read_csv("../data/sample_emails_with_triage_200.csv")

results = []
for _, row in emails.head(20).iterrows():
    email = row["body"]
    output = app.invoke({"email": email})
    print(output["triage"])
    results.append({
        "email": email,
        "triage": output["triage"],
        "response": output.get("response", "")
    })

pd.DataFrame(results).to_csv("../data/milestone1_output.csv", index=False)


In [ ]:
import pandas as pd

emails = pd.read_csv("../data/sample_emails_with_triage_200.csv")

gold = pd.DataFrame({
    "email": emails.head(25)["body"],
    "expected": [""] * 25
})

gold.to_csv("../data/golden_labels.csv", index=False)


In [ ]:
gold = pd.read_csv("../data/golden_labels.csv")
pred = pd.read_csv("../data/milestone1_output.csv")

gold = gold.reset_index(drop=True)
pred = pred.reset_index(drop=True)

accuracy = (gold["expected"] == pred["triage"]).mean()
accuracy


0.9